linear
ridge
lasso
dtr
knn with std scaler

top : GBMR

In [18]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, StackingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, r2_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from tqdm import tqdm
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

In [14]:
df = pd.read_csv("insurance (12).csv")
df

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


In [15]:
X,y = df.drop("charges",axis=1), df["charges"]
X_train, X_test, y_train, y_test = train_test_split(X,y,random_state=26,test_size=0.3)

In [5]:
ohc = OneHotEncoder(sparse_output=False, drop="first").set_output(transform="pandas")
ct = make_column_transformer((ohc,make_column_selector(dtype_include=object)),
                             ("passthrough", make_column_selector(dtype_include=['int64','float64'])),
                             verbose_feature_names_out=False)
ct = ct.set_output(transform='pandas')

In [19]:
scaler = MinMaxScaler()

In [20]:
lr = LinearRegression()
rd = Ridge()
la = Lasso()
dtr = DecisionTreeRegressor(random_state=26)
gbm = GradientBoostingRegressor(random_state=26)
knn = KNeighborsRegressor()
pipe_knn = Pipeline([("SCL",scaler),("KNN",knn)])

In [23]:
stack = StackingRegressor(estimators=[("lr",lr),("rd",rd),("la",la),("KNN",pipe_knn),("TREE",dtr)],
                           final_estimator=gbm)
pipe = Pipeline([("OHE",ct),("stack",stack)])
pipe.fit(X_train,y_train)
y_pred = pipe.predict(X_test)
r2_score(y_test,y_pred)

0.8507601741710034

In [29]:
scaler = StandardScaler()
lr = LinearRegression()
rd = Ridge()
la = Lasso()
dtr = DecisionTreeRegressor(random_state=26)
gbm = GradientBoostingRegressor(random_state=26)
knn = KNeighborsRegressor()
pipe_knn = Pipeline([("SCL",scaler),("KNN",knn)])
stack = StackingRegressor(estimators=[("lr",lr),("rd",rd),("la",la),("KNN",pipe_knn),("TREE",dtr)],
                           final_estimator=gbm)
pipe = Pipeline([("OHE",ct),("stack",stack)])
pipe.fit(X_train,y_train)
y_pred = pipe.predict(X_test)
r2_score(y_test,y_pred)

0.853889441706591

--------------

In [27]:
scaler = StandardScaler()
lr = LinearRegression()
rd = Ridge()
la = Lasso()
dtr = DecisionTreeRegressor(random_state=26)
gbm = GradientBoostingRegressor(random_state=26)
knn = KNeighborsRegressor()
stack = StackingRegressor(estimators=[("lr",lr),("rd",rd),("la",la),("KNN",pipe_knn),("TREE",dtr)],
                           final_estimator=gbm)
pipe = Pipeline([("OHE",ct),("Scaler",scaler),("stack",stack)])
pipe.fit(X_train,y_train)
y_pred = pipe.predict(X_test)
r2_score(y_test,y_pred)

0.8514311613464224

-----------

In [28]:
scaler = MinMaxScaler()
lr = LinearRegression()
rd = Ridge()
la = Lasso()
dtr = DecisionTreeRegressor(random_state=26)
gbm = GradientBoostingRegressor(random_state=26)
knn = KNeighborsRegressor()
stack = StackingRegressor(estimators=[("lr",lr),("rd",rd),("la",la),("KNN",pipe_knn),("TREE",dtr)],
                           final_estimator=gbm)
pipe = Pipeline([("OHE",ct),("Scaler",scaler),("stack",stack)])
pipe.fit(X_train,y_train)
y_pred = pipe.predict(X_test)
r2_score(y_test,y_pred)

0.8524539570260611

-----------

In [30]:
pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('OHE', ...), ('stack', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['age','sex','bmi','children','smoker','region']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('onehotencoder', ...), ('passthrough', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error i

In [31]:
scaler = MinMaxScaler()
lr = LinearRegression()
rd = Ridge()
la = Lasso()
dtr = DecisionTreeRegressor(random_state=26)
gbm = GradientBoostingRegressor(random_state=26)
knn = KNeighborsRegressor()
stack = StackingRegressor(estimators=[("lr",lr),("rd",rd),("la",la),("KNN",pipe_knn),("TREE",dtr)],
                           final_estimator=gbm,passthrough=True)
pipe = Pipeline([("OHE",ct),("Scaler",scaler),("stack",stack)])
pipe.fit(X_train,y_train)
y_pred = pipe.predict(X_test)
r2_score(y_test,y_pred)

0.8580218265379616

In [34]:
scaler = MinMaxScaler()
lr = LinearRegression()
rd = Ridge()
la = Lasso()
dtr = DecisionTreeRegressor(random_state=26)
gbm = XGBRegressor(random_state=26)
knn = KNeighborsRegressor()
stack = StackingRegressor(estimators=[("lr",lr),("rd",rd),("la",la),("KNN",pipe_knn),("TREE",dtr)],
                           final_estimator=gbm,passthrough=True)
pipe = Pipeline([("OHE",ct),("Scaler",scaler),("stack",stack)])
pipe.fit(X_train,y_train)
y_pred = pipe.predict(X_test)
r2_score(y_test,y_pred)

0.8361569554287581